In [97]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import glob
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2

## Importing labels and images into variables

In [98]:
home = "images"
directories = "train", "test", "val"

In [99]:
def read_file_to_list(file_path):
    with open(file_path, 'r') as file:
        content = file.read().strip()
    return [int(char) for char in content]

In [100]:
y_train = read_file_to_list(f'{home}/{directories[0]}.txt')
y_test = read_file_to_list(f'{home}/{directories[1]}.txt')
y_val = read_file_to_list(f'{home}/{directories[2]}.txt')

In [101]:
train_folder = os.path.join(home, directories[0])
test_folder = os.path.join(home, directories[1])
val_folder = os.path.join(home, directories[2])

def get_image_content(folder_path):
    image_contents = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if os.path.isfile(file_path):
            image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            image_contents.append(image)
    return image_contents

x_train = get_image_content(train_folder)
x_test = get_image_content(test_folder)
x_val = get_image_content(val_folder)

## Training the CNN

In [102]:
x_train = np.array(x_train).reshape((-1, 224, 224, 1)).astype('float32')
x_val = np.array(x_val).reshape((-1, 224, 224, 1)).astype('float32')
x_test = np.array(x_test).reshape((-1, 224, 224, 1)).astype('float32')

y_train = tf.keras.utils.to_categorical(np.array(y_train), num_classes=4)
y_val = tf.keras.utils.to_categorical(np.array(y_val), num_classes=4)
y_test = tf.keras.utils.to_categorical(np.array(y_test), num_classes=4)

In [114]:
tf.random.set_seed(42)
model = tf.keras.Sequential()
input_shape = (224, 224, 1)
model.add(tf.keras.layers.Conv2D(10, 3, activation="relu", input_shape=input_shape))
model.add(tf.keras.layers.MaxPool2D())
model.add(tf.keras.layers.Conv2D(10, 3, activation="relu"))
model.add(tf.keras.layers.MaxPool2D())
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(4, activation="softmax"))

model.compile(loss=tf.keras.losses.CategoricalCrossentropy(),
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

history = model.fit(x=x_train, y=y_train, epochs=30, validation_data=(x_val, y_val), batch_size=16)

test_loss, test_acc = model.evaluate(x_test, y_test)

Epoch 1/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.2424 - loss: 81.7216 - val_accuracy: 0.2630 - val_loss: 1.4698
Epoch 2/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.2618 - loss: 1.4148 - val_accuracy: 0.2464 - val_loss: 1.4393
Epoch 3/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.2692 - loss: 1.3857 - val_accuracy: 0.2393 - val_loss: 1.4352
Epoch 4/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.2946 - loss: 1.3684 - val_accuracy: 0.2417 - val_loss: 1.4535
Epoch 5/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.3204 - loss: 1.3522 - val_accuracy: 0.2512 - val_loss: 1.4695
Epoch 6/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.3529 - loss: 1.3314 - val_accuracy: 0.2654 - val_loss: 1.4676
Epoch 7/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.3811 - loss: 1.3075 - val_accuracy: 0.2607 - val_loss: 1.4992
Epoch 8/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.4059 - loss: 1.2794 - val_acc

Beginning values look as if it was about to become an overfit model, will try to work on it but the more epochs the better the results.
Will perform data augmentation and use dropout to try to compensate it.